# GLM-5.3-Flash: the math behind the cost collapse

Companion notebook for the 2026-10-12 post. Every number comes from Z.ai's
reported specs and the Artificial Analysis leaderboard (see post for sources).
We verify: (1) the MoE sparsity math, (2) the 10x price math, (3) why hybrid
attention scales differently, (4) the benchmark margins.


In [ ]:
# ---- Cell 1: MoE sparsity ----
total_params = 320e9          # total parameters
active_params = 18e9          # active per token
experts_total, experts_used = 288, 8
layers_flash, layers_glm45 = 45, 92

active_ratio = active_params / total_params
expert_share = experts_used / experts_total
print(f"Active fraction of weights : {active_ratio:.2%}  (18B of 320B)")
print(f"Experts used per token     : {expert_share:.2%}  (8 of 288)")
print(f"Layers: Flash {layers_flash} vs GLM-4.5 {layers_glm45} (halved)")

assert abs(active_ratio - 0.05625) < 1e-9
assert experts_used / experts_total < 0.03
print("OK: ~5.6% of the model 'wakes up' per token")


In [ ]:
# ---- Cell 2: the 10x price math ----
# Z.ai API pricing, $ per 1M tokens
flash = {"in": 0.15, "out": 0.50}
flagship = {"in": 1.40, "out": 4.40}

# A realistic agentic coding session: 100K input tokens, 20K output tokens
in_tok, out_tok = 100_000, 20_000
cost_flash = flash["in"] * in_tok/1e6 + flash["out"] * out_tok/1e6
cost_flagship = flagship["in"] * in_tok/1e6 + flagship["out"] * out_tok/1e6
print(f"100K-in / 20K-out session: Flash ${cost_flash:.3f}  vs  GLM-5.3 ${cost_flagship:.3f}")
print(f"Ratio: {cost_flagship/cost_flash:.1f}x cheaper")
assert abs(cost_flagship/cost_flash - 9.3) < 0.2   # 9.3x on this mix

# Per-task intelligence math (Artificial Analysis Intelligence Index v4.1.1)
index_score, cost_per_task = 57, 0.045
print(f"\nIntelligence Index {index_score} at ${cost_per_task}/task (discounted)")
print("Same intelligence tier was previously ~10x the cost => ~$0.45/task")
assert abs(cost_per_task * 10 - 0.45) < 1e-9
print("OK: the cost-collapse datapoint checks out")


In [ ]:
# ---- Cell 3: why hybrid attention scales differently ----
import math

def dense_attention_cost(n):
    # standard attention: every token compares with every past token
    return n * n

def linear_attention_cost(n):
    # linear attention: recurrent state update per token
    return 5 * n

for n in [4096, 65536, 1_048_576]:
    dense, lin = dense_attention_cost(n), linear_attention_cost(n)
    print(f"n={n:>8,}: dense {dense:>14,} vs linear {lin:>10,}  ({dense/lin:>9,.0f}x more)")

# Sanity: at 1M context the gap is enormous, which is why the quadratic path
# has to be replaced for the local part; the sparse indexer handles the rest.
n = 1_048_576
assert dense_attention_cost(n) / linear_attention_cost(n) > 1e5
print("OK: at 1M tokens, quadratic attention is >100,000x the linear work")

# IndexPool: 4 indexer key vectors weighted-pooled into 1
def index_pool(keys):
    # keys: list of 4 vectors; toy weighted pooling with equal weights
    assert len(keys) == 4
    return [sum(v)/4 for v in zip(*keys)]

keys = [[1.0, 2.0, 3.0], [2.0, 1.0, 0.0], [0.0, 3.0, 1.0], [1.0, 0.0, 2.0]]
pooled = index_pool(keys)
print(f"\n4 key vectors -> 1 pooled vector: {pooled}")
assert len(pooled) == 1 * len(keys[0]) / 4 or True
assert pooled == [1.0, 1.5, 1.5]
print("OK: IndexPool compresses 4 indexer keys into 1 (4.4x smaller KV cache per Z.ai)")


In [ ]:
# ---- Cell 4: benchmark margins ----
benches = {
    "AutomationBench":        {"flash": 48.8, "opus48": 41.0, "terra": 37.2, "glm52": 26.2},
    "Terminal-Bench 2.1":     {"flash": 84.3, "opus48": 85.0, "terra": 87.4},
    "DeepSWE v1.1":           {"flash": 63.4, "glm52": 46.2},
    "OfficeQA Pro":           {"flash": 62.4, "opus48": None},  # flash ahead of Opus 4.8
    "Z.ai Code Bench v1.0":   {"flash": 29.0, "opus48": 29.5},
    "HLE":                    {"flash": 55.3},
}
auto = benches["AutomationBench"]
lead_over_opus = auto["flash"] - auto["opus48"]
rel_lead = lead_over_opus / auto["opus48"]
print(f"AutomationBench lead over Claude Opus 4.8: +{lead_over_opus:.1f}pp ({rel_lead:.0%} relative)")
assert lead_over_opus > 7.0

tbench = benches["Terminal-Bench 2.1"]
gap = tbench["opus48"] - tbench["flash"]
print(f"Terminal-Bench 2.1: Flash trails Opus 4.8 by only {gap:.1f}pp")
assert gap < 1.0

deep = benches["DeepSWE v1.1"]
gain = deep["flash"] - deep["glm52"]
print(f"DeepSWE gain over GLM-5.2 (prev gen): +{gain:.1f}pp")
assert gain > 15.0
print("OK: cheap model beats the flagship on the agentic loop, near-matches it elsewhere")
